# E4b · RoBERTa (GPU) · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.4, RQ3)
Mốc tham chiếu học sâu cho **T5.6**. Hộp thời gian: chưa xong trước tối **05/11** thì bỏ, chỉ báo cáo LR và NB.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**, rồi Runtime → Run all.
**Chạy sau E4** (cần file T5.6_model_compare.csv do E4 tạo).

**Kỷ luật thí nghiệm (giống E1–E4):** huấn luyện trên tập con của `clean_train`; chọn ngưỡng trên `clean_val` (FPR 5%); `clean_test` chỉ dùng để báo cáo, chạy **một lần**.

**Điểm số = log-odds (logit AI − logit người)**, không dùng xác suất softmax: mô hình fine-tune rất tự tin, xác suất bão hòa đúng 1,0 như Naive Bayes ở E4. Ranh giới mặc định tương ứng p = 0,5 là log-odds = 0.

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son&Tai"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

In [ ]:
# 1. Dữ liệu: toàn bộ người trong train + 10.000 AI (cố định random_state) → ~19.650 văn bản
import numpy as np, torch
from aitext import evaluate
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"
GPU = torch.cuda.get_device_name(0); print(GPU)
COLS = ["generation", "label", "model", "domain"]
tr = pd.read_parquet(config.SPLIT_TRAIN, columns=COLS)
va = pd.read_parquet(config.SPLIT_VAL, columns=COLS)
te = pd.read_parquet(config.SPLIT_TEST, columns=COLS)
N_AI = 10_000
sub = pd.concat([tr[tr.label == 0], tr[tr.label == 1].sample(N_AI, random_state=config.SEED)]
                ).sample(frac=1, random_state=config.SEED).reset_index(drop=True)
print("train con:", sub.label.value_counts().to_dict(), "| val:", len(va), "| test:", len(te))

In [ ]:
# 2. Fine-tune roberta-base: 1 epoch, max_length 256, fp16 (~10–20 phút trên T4)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
torch.manual_seed(config.SEED)
NAME, MAXLEN, BS, LR_ = "roberta-base", 256, 16, 2e-5
tok = AutoTokenizer.from_pretrained(NAME)
net = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=2).cuda()
opt = torch.optim.AdamW(net.parameters(), lr=LR_, weight_decay=0.01)
steps = (len(sub) + BS - 1) // BS
sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.cuda.amp.GradScaler()

def enc(texts):
    return {k: v.cuda() for k, v in tok(list(texts), truncation=True, max_length=MAXLEN,
                                        padding=True, return_tensors="pt").items()}

net.train(); t0 = time.time()
for i in range(steps):
    b = sub.iloc[i * BS:(i + 1) * BS]
    with torch.autocast("cuda", dtype=torch.float16):
        loss = net(**enc(b.generation), labels=torch.tensor(b.label.values).cuda()).loss
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
    if i % 200 == 0: print(f"bước {i}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s")
train_s = time.time() - t0; print(f"Huấn luyện xong: {train_s:.0f}s")

In [ ]:
# 3. Chấm điểm = log-odds; hàm đo y hệt evaluate.metrics (ngưỡng nghiêm ngặt "> thr")
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def score(df, bs=64):
    net.eval(); out = []
    for i in range(0, len(df), bs):
        with torch.autocast("cuda", dtype=torch.float16):
            lg = net(**enc(df.generation.iloc[i:i + bs])).logits.float()
        out.append((lg[:, 1] - lg[:, 0]).cpu().numpy())
    return np.concatenate(out)

def thr_at_fpr(s, y, fpr=config.FPR_TARGET): return float(np.quantile(s[y == 0], 1 - fpr))
def rate_above(s, y, label, thr): return float((s[y == label] > thr).mean())
def metrics(s, y, thr):
    return {"auroc": roc_auc_score(y, s), "fpr_at_0.5": rate_above(s, y, 0, 0.0),   # log-odds 0 ≡ p 0,5
            "tpr_at_fpr_oracle": rate_above(s, y, 1, thr_at_fpr(s, y)), "n": len(y),
            "threshold": thr, "tpr": rate_above(s, y, 1, thr), "fpr": rate_above(s, y, 0, thr)}

s_va = score(va); thr = thr_at_fpr(s_va, va.label.values)
print(f"Ngưỡng chọn trên val (log-odds): {thr:.3f}  | số điểm val trùng nhau: {len(s_va) - len(np.unique(s_va))}")

## 4. Đo trên TEST: chạy **một lần**, sau khi đã chốt mọi thứ ở trên

In [ ]:
t0 = time.time(); s_te = score(te); infer_s = time.time() - t0
r = metrics(s_te, te.label.values, thr)
evaluate.log_run("E4b", OWNER, "roberta|clean|19.6k", r,
                 notes=f"train {train_s:.0f}s + test {infer_s:.0f}s, {GPU}, roberta-base 1 epoch maxlen {MAXLEN}, điểm log-odds")
pd.DataFrame({"label": te.label.values, "model": te.model.values, "domain": te.domain.values, "score": s_te}
             ).to_parquet(f"{config.RESULTS_DIR}/E4b_roberta_test_scores.parquet", index=False)   # lưu vết để kiểm tra lại

# Thêm 1 dòng vào T5.6 (giữ nguyên các dòng LR, NB)
t56 = pd.read_csv(f"{config.RESULTS_DIR}/T5.6_model_compare.csv")
t56 = t56[t56["mô hình"] != "roberta"]
row = {"mô hình": "roberta", "điểm": "log-odds", "train (s)": train_s, "phần cứng": f"Colab {GPU}", **r}
t56 = pd.concat([t56, pd.DataFrame([row])], ignore_index=True)
display(t56.round(4))
evaluate.save_table(t56, "T5.6_model_compare",
                    notes="roberta-base huấn luyện trên ~19.650 văn bản (toàn bộ người trong train + 10.000 AI), 1 epoch, GPU; "
                          "LR và NB huấn luyện trên toàn bộ train bằng Spark trên CPU. Thời gian không so trực tiếp được.")

In [ ]:
# 5. AUROC theo LLM: đối chiếu với E1/E3 (mô hình base có còn khó bắt với RoBERTa không?)
h = s_te[te.label.values == 0]; rows = []
for g_ in config.OLD_GENERATORS + config.NEW_GENERATORS:
    a = s_te[(te.model == g_).values]
    rows.append({"LLM": g_, "AUROC": roc_auc_score(np.r_[np.zeros(len(h)), np.ones(len(a))], np.r_[h, a]),
                 "TPR": float((a > thr).mean())})
t56b = pd.DataFrame(rows); display(t56b.round(3)); evaluate.save_table(t56b, "T5.6b_roberta_by_llm")

## 6. (Tùy chọn, ~3 phút) RoBERTa trước tấn công ký tự
Mô hình học sâu dùng tokenizer BPE nên ký tự Cyrillic và zero-width cũng làm vỡ token. Ô này cho biết RoBERTa có cần chuẩn hóa như TF-IDF không.

In [ ]:
att = pd.read_parquet(config.ATTACKED_DIR, columns=COLS + ["attack"]); rows = []
for atk in ["homoglyph", "zero_width_space"]:
    d = att[att.attack.astype(str) == atk]
    s = score(d); y = d.label.values
    rows.append({"attack": atk, "AUROC": roc_auc_score(y, s), "TPR": rate_above(s, y, 1, thr), "FPR": rate_above(s, y, 0, thr)})
t56c = pd.DataFrame(rows); display(t56c.round(3)); evaluate.save_table(t56c, "T5.6c_roberta_char_attacks")

## Ghi chú cho Chương 5
- ...